# 任务三：真实 Qwen 全参数微调的显存预算实验（Colab）

在 Colab 中选择 **运行时 → 更改运行时类型 → T4 GPU**，然后从上到下执行。首次执行需要联网下载真实权重和 WikiText。这里只准备实验，不包含任何预填结果。

默认 Qwen/Qwen2.5-0.5B、序列长度 128、有效 batch 2，比较 baseline（micro 2）、累积（micro 1 × 2）、checkpoint（micro 2）。每种策略从同一个固定提交重新加载 FP32 权重，使用相同的预定输入和 AdamW；仅 autocast 使用 BF16/FP16，**不是量化、LoRA 或半精度权重训练**。FP32 参数、梯度与 Adam 的两个状态合计约 8 GB（十进制；精确值由参数量计算），另有激活、logits、CUDA 上下文和临时分配。默认适合尝试免费 T4，但不承诺所有 Colab 环境都能容纳。

预热 3 个更新 + 测量 5 个更新只是短跑比较，不是训练收敛证据。预热会更新参数，所有策略都执行同样的预热批次。验证来自独立 validation split，同一组样本在训练前后评估。下载、初始化和验证不计入训练计时或训练峰值。OOM 会拒绝该策略，绝不偷偷缩小 batch 或序列。


In [ ]:
# 安装固定兼容版本，不安装或替换 Colab 的 torch / torchvision / CUDA。
%pip -q install "bitsandbytes>=0.43.0" "transformers==4.48.3" "datasets==3.2.0" "huggingface_hub==0.28.1" "accelerate==1.3.0" "matplotlib==3.10.0" "safetensors>=0.4.3"
# 若当前内核此前已导入不同版本的这些库，请先重启运行时，再从此处执行。


In [ ]:
import os, gc, json, math, time, hashlib, random, statistics, shutil, csv
from pathlib import Path
from importlib.metadata import version
import numpy as np
import torch
assert torch.cuda.is_available(), "需要 GPU：请选择 Colab 的 T4 GPU 运行时，不能在 CPU 上继续。"
from transformers import AutoConfig, AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset
from huggingface_hub import HfApi, snapshot_download
import matplotlib.pyplot as plt

GIB = 1024 ** 3
DEVICE = torch.device("cuda:0")
torch.cuda.set_device(DEVICE)
AMP_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
ENV = {k: version(k) for k in ["torch", "transformers", "datasets", "huggingface_hub", "accelerate", "matplotlib", "safetensors"]}
assert ENV["transformers"] == "4.48.3", "请重启内核以使用刚安装的版本。"
ENV.update(gpu=torch.cuda.get_device_name(0), cuda=torch.version.cuda,
           total_gib=torch.cuda.get_device_properties(0).total_memory/GIB,
           autocast=str(AMP_DTYPE), weight_dtype="torch.float32")
print(json.dumps(ENV, ensure_ascii=False, indent=2))


## 配置与判定规则

预算以**测量阶段训练的 peak reserved** 加固定安全余量判定，同时报告 peak allocated，不能把两者混淆。预算上限还受本次 GPU 总容量约束。预估检查只提供开跑前下限，最终以真实峰值为准；PyTorch 统计不覆盖所有驱动/外部进程显存。

PASS 必须满足：固定提交/配置/初始权重子集摘要/训练与验证输入一致；所有损失有限；全部更新成功；初始验证 loss 差不超过容差；最终验证 loss 相对 baseline 差不超过显式容差；吞吐至少达到 baseline 的指定比例；峰值加余量不超预算。baseline 不可用时其他策略不能 PASS。最后验证 loss 不代表长期质量，也不要求短跑一定下降。


In [ ]:
CFG = dict(seed=73, model_id="Qwen/Qwen2.5-0.5B", model_revision="main",
           dataset_id="Salesforce/wikitext", dataset_config="wikitext-2-raw-v1",
           dataset_revision="main", seq_len=128, effective_batch=2,
           warmup_steps=3, measured_steps=5, eval_chunks=8,
           lr=1e-5, weight_decay=0.01, max_grad_norm=1.0,
           budget_gib=14.0, safety_margin_gib=1.0,
           min_throughput_ratio=0.40, initial_loss_atol=1e-5,
           final_loss_atol=0.15, run_profiler=False)
# 可增大步数，但这会改变工作负载。修改后必须从本单元重新执行所有策略。
assert CFG["effective_batch"] == 2 and CFG["seq_len"] >= 2
assert CFG["warmup_steps"] >= 1 and CFG["measured_steps"] >= 1
assert CFG["eval_chunks"] >= 1 and CFG["safety_margin_gib"] > 0
assert 0 < CFG["min_throughput_ratio"] <= 1
CASES = [("baseline", 2, False), ("accumulation", 1, False), ("checkpoint", 2, True)]
OUT = Path("/content/task3_real_model_artifacts")
OUT.mkdir(parents=True, exist_ok=True)
# 重跑不覆盖旧证据：每次配置使用新的目录。
RUN = OUT / time.strftime("%Y%m%d_%H%M%S")
RUN.mkdir(exist_ok=False)
def save_json(name, value):
    (RUN / name).write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")
def seed_all():
    random.seed(CFG["seed"])
    np.random.seed(CFG["seed"])
    torch.manual_seed(CFG["seed"])
    torch.cuda.manual_seed_all(CFG["seed"])
def sync():
    torch.cuda.synchronize(DEVICE)
def cleanup():
    gc.collect()
    torch.cuda.empty_cache()
    sync()
seed_all()
save_json("environment.json", ENV)
save_json("config.json", CFG)
print("证据目录：", RUN)


In [ ]:
# 首次解析 main 为不可变提交；下载后所有策略只从该本地快照加载。
api = HfApi()
MODEL_SHA = api.model_info(CFG["model_id"], revision=CFG["model_revision"]).sha
DATA_SHA = api.dataset_info(CFG["dataset_id"], revision=CFG["dataset_revision"]).sha
assert len(MODEL_SHA) == 40 and len(DATA_SHA) == 40
MODEL_PATH = snapshot_download(CFG["model_id"], revision=MODEL_SHA,
                               allow_patterns=["*.json", "*.safetensors", "*.txt", "*.model"])
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, local_files_only=True)
model_config = AutoConfig.from_pretrained(MODEL_PATH, local_files_only=True)
raw = load_dataset(CFG["dataset_id"], CFG["dataset_config"], revision=DATA_SHA,
                   split={"train": "train", "validation": "validation"})
PROVENANCE = dict(model_id=CFG["model_id"], model_commit=MODEL_SHA,
                  dataset_id=CFG["dataset_id"], dataset_commit=DATA_SHA,
                  dataset_fingerprints={k: v._fingerprint for k, v in raw.items()},
                  model_config=model_config.to_dict())
save_json("provenance.json", PROVENANCE)
print("模型提交：", MODEL_SHA, "数据提交：", DATA_SHA)
print({k: len(v) for k, v in raw.items()})


In [ ]:
# 连续打包为等长块；段落之间插入 EOS；不填充，所以每个标签均有效。
# 如果改成 padding 数据管线，必须把 padding 位置标签改为 -100，而非用 EOS 判定 padding。
def pack_split(split, n_chunks):
    target = n_chunks * CFG["seq_len"]
    ids = []
    for text in split["text"]:
        if not text.strip():
            continue
        ids.extend(tokenizer.encode(text, add_special_tokens=False))
        ids.append(tokenizer.eos_token_id)
        if len(ids) >= target:
            break
    assert len(ids) >= target, "数据不足；不能复用训练数据做验证。"
    return torch.tensor(ids[:target], dtype=torch.long).reshape(n_chunks, CFG["seq_len"])
N_STEPS = CFG["warmup_steps"] + CFG["measured_steps"]
train_cpu = pack_split(raw["train"], N_STEPS * CFG["effective_batch"])
eval_cpu = pack_split(raw["validation"], CFG["eval_chunks"])
# 批次在 CPU 上提前固定；每种策略第 k 次更新读取同一 batch。
train_batches = train_cpu.reshape(N_STEPS, CFG["effective_batch"], CFG["seq_len"])
def tensor_sha(t):
    return hashlib.sha256(t.contiguous().numpy().tobytes()).hexdigest()
INPUTS = dict(train_sha256=tensor_sha(train_cpu), validation_sha256=tensor_sha(eval_cpu),
              batch_sha256=[tensor_sha(b) for b in train_batches],
              train_shape=list(train_batches.shape), validation_shape=list(eval_cpu.shape),
              padding=False, causal_targets_per_update=CFG["effective_batch"]*(CFG["seq_len"]-1))
torch.save({"train_batches": train_batches, "validation": eval_cpu}, RUN / "fixed_inputs.pt")
save_json("inputs.json", INPUTS)
del raw
print(INPUTS)


## 执行函数

FP16 使用 GradScaler，先缩放反传，再 unscale、检查有限梯度、裁剪、step、update；如果 scale 下降或梯度不有限，本策略立即拒绝，不把跳过更新当成功。BF16 不启用 scaler。每个 micro loss 按 micro/effective batch 归一化；由于所有块等长且无 padding，这等价于对有效 token 平均。

每步起止都有 CUDA synchronize，计时包含 CPU→GPU 搬运、前反传、梯度检查、优化器更新和同步，不包含模型加载与验证。预热结束清梯度并重置峰值；各测量步独立重置峰值，策略峰值取这些步最大值。allocated/reserved 的当前值也逐步保存。验证前清梯度；验证显存绝不冒充训练峰值。初始化摘要覆盖每个参数的首尾各 16 个 FP32 数值及名称/形状，**不是完整权重的密码学证明**；不可变模型 SHA、本地固定快照与配置摘要共同用于检查初态。


In [ ]:
def initial_identity(model):
    h = hashlib.sha256()
    for name, p in model.named_parameters():
        h.update(name.encode())
        h.update(str(tuple(p.shape)).encode())
        a = p.detach().reshape(-1)
        h.update(torch.cat((a[:16], a[-16:])).float().cpu().numpy().tobytes())
    config_text = json.dumps(model.config.to_dict(), sort_keys=True)
    return dict(model_commit=MODEL_SHA, subset_sha256=h.hexdigest(),
                config_sha256=hashlib.sha256(config_text.encode()).hexdigest(),
                train_sha256=tensor_sha(train_cpu), validation_sha256=tensor_sha(eval_cpu))

def make_model(checkpoint_enabled):
    seed_all()
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_PATH, local_files_only=True, torch_dtype=torch.float32,
        attn_implementation="sdpa", low_cpu_mem_usage=True)
    model.config.use_cache = False
    model.requires_grad_(True)
    # 在 CPU 上做静态预算检查，避免明知无法容纳仍先把参数放到 GPU。
    n = sum(p.numel() for p in model.parameters())
    cleanup()
    free, total = torch.cuda.mem_get_info()
    limit = min(CFG["budget_gib"], total / GIB)
    static_gib = n * 16 / GIB  # FP32 参数+梯度+Adam m/v；不含激活、logits 等。
    budget = dict(parameters=n, static_training_gib=static_gib,
                  free_before_model_gib=free/GIB, budget_gib=limit)
    if static_gib + CFG["safety_margin_gib"] > min(limit, free/GIB):
        del model
        raise BudgetRejected(budget)
    ident = initial_identity(model)
    if checkpoint_enabled:
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    model.to(DEVICE)
    assert all(p.dtype == torch.float32 and p.requires_grad for p in model.parameters())
    return model, ident, budget

class BudgetRejected(RuntimeError):
    def __init__(self, budget):
        super().__init__("静态状态下限加安全余量已超过预算或可用显存")
        self.budget = budget

@torch.no_grad()
def evaluate(model, optimizer=None):
    if optimizer is not None:
        optimizer.zero_grad(set_to_none=True)
    assert all(p.grad is None for p in model.parameters()), "验证前必须清理训练梯度"
    model.eval()
    total_loss, total_targets = 0.0, 0
    for row in eval_cpu:
        x = row.unsqueeze(0).to(DEVICE)
        with torch.autocast("cuda", dtype=AMP_DTYPE):
            loss = model(input_ids=x, attention_mask=torch.ones_like(x), labels=x, use_cache=False).loss
        n = x.shape[0] * (x.shape[1]-1)
        total_loss += float(loss) * n
        total_targets += n
        del x, loss
    sync()
    model.train()
    return total_loss / total_targets

def train_step(model, optimizer, scaler, batch, micro, effective_batch=None):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    sync()
    torch.cuda.reset_peak_memory_stats(DEVICE)
    start = time.perf_counter()
    eff = CFG["effective_batch"] if effective_batch is None else effective_batch
    losses = []
    for offset in range(0, eff, micro):
        x = batch[offset:offset+micro].to(DEVICE)
        with torch.autocast("cuda", dtype=AMP_DTYPE):
            loss = model(input_ids=x, attention_mask=torch.ones_like(x), labels=x, use_cache=False).loss
            normalized = loss * (x.shape[0] / eff)
        scaler.scale(normalized).backward()
        losses.append(loss.detach() * (x.shape[0] / eff))
        del x, loss, normalized
    scaler.unscale_(optimizer)
    norm = torch.nn.utils.clip_grad_norm_(model.parameters(), CFG["max_grad_norm"], foreach=False)
    finite = bool(torch.isfinite(norm).item())
    old_scale = scaler.get_scale()
    # 非有限梯度或范数：不调用优化器；scaler.update 仍消费 unscale 的检查记录。
    if finite:
        scaler.step(optimizer)
    scaler.update()
    skipped = (not finite) or scaler.get_scale() < old_scale
    mean_loss = float(torch.stack(losses).sum())
    del losses, norm
    sync()
    elapsed = time.perf_counter() - start
    return dict(loss=mean_loss, seconds=elapsed,
                input_tokens_per_second=eff*batch.shape[-1]/elapsed,
                target_tokens_per_second=eff*(batch.shape[-1]-1)/elapsed,
                peak_allocated_gib=torch.cuda.max_memory_allocated(DEVICE)/GIB,
                peak_reserved_gib=torch.cuda.max_memory_reserved(DEVICE)/GIB,
                allocated_gib=torch.cuda.memory_allocated(DEVICE)/GIB,
                reserved_gib=torch.cuda.memory_reserved(DEVICE)/GIB,
                scale_before=old_scale, scale_after=scaler.get_scale(), skipped_update=skipped)

def run_case(name, micro, checkpoint_enabled, profile_run=False):
    model = optimizer = scaler = prof = None
    result = dict(strategy=name, micro_batch=micro, accumulation=CFG["effective_batch"]//micro,
                  checkpoint=checkpoint_enabled, status="RUNNING", steps=[], profile=profile_run)
    try:
        cleanup()
        model, ident, budget = make_model(checkpoint_enabled)
        result.update(identity=ident, budget=budget)
        optimizer = torch.optim.AdamW(model.parameters(), lr=CFG["lr"],
                                     weight_decay=CFG["weight_decay"], foreach=False)
        scaler = torch.amp.GradScaler("cuda", enabled=AMP_DTYPE == torch.float16, init_scale=1024.0)
        result["initial_validation_loss"] = evaluate(model, optimizer)
        if not math.isfinite(result["initial_validation_loss"]):
            result["status"] = "REJECT_NONFINITE"
            return result
        cleanup()  # 初始验证已结束，释放其空闲缓存；不在各训练步间清缓存。
        for i, batch in enumerate(train_batches):
            if i == CFG["warmup_steps"]:
                optimizer.zero_grad(set_to_none=True)
                sync()
                torch.cuda.reset_peak_memory_stats(DEVICE)
            # profiler 单独运行，只抓取预热后第一个更新，不参与性能/预算比较。
            if profile_run and i == CFG["warmup_steps"]:
                with torch.profiler.profile(activities=[torch.profiler.ProfilerActivity.CPU,
                                                        torch.profiler.ProfilerActivity.CUDA],
                                            record_shapes=True, profile_memory=True, with_stack=False) as prof:
                    record = train_step(model, optimizer, scaler, batch, micro)
                prof.export_chrome_trace(str(RUN / "checkpoint_trace.json"))
                prof = None
            else:
                record = train_step(model, optimizer, scaler, batch, micro)
            record.update(step=i, phase="warmup" if i < CFG["warmup_steps"] else "measured")
            result["steps"].append(record)
            print(name, record)
            if record["skipped_update"] or not math.isfinite(record["loss"]):
                result["status"] = "REJECT_SKIPPED_OR_NONFINITE"
                return result
            if profile_run and i == CFG["warmup_steps"]:
                break
        measured = [r for r in result["steps"] if r["phase"] == "measured"]
        result.update(train_peak_allocated_gib=max(r["peak_allocated_gib"] for r in measured),
                      train_peak_reserved_gib=max(r["peak_reserved_gib"] for r in measured),
                      measured_seconds=sum(r["seconds"] for r in measured),
                      target_tokens_per_second=len(measured)*INPUTS["causal_targets_per_update"]/sum(r["seconds"] for r in measured),
                      step_seconds_median=statistics.median(r["seconds"] for r in measured),
                      step_seconds_stdev=statistics.stdev([r["seconds"] for r in measured]) if len(measured)>1 else 0.0)
        optimizer.zero_grad(set_to_none=True)
        result["final_validation_loss"] = evaluate(model, optimizer)
        result["status"] = "COMPLETE" if math.isfinite(result["final_validation_loss"]) else "REJECT_NONFINITE"
        return result
    except torch.cuda.OutOfMemoryError as exc:
        result.update(status="REJECT_OOM", reason=str(exc))
        return result
    except BudgetRejected as exc:
        result.update(status="REJECT_STATIC_BUDGET", reason=str(exc), budget=exc.budget)
        return result
    finally:
        # 未列明的异常直接抛出。OOM 不改变任何后续策略的工作负载。
        if optimizer is not None:
            optimizer.zero_grad(set_to_none=True)
        model = optimizer = scaler = prof = None
        cleanup()
        # 即使中途失败，也留下已取得的逐步证据；JSON 禁止非标准 NaN/Infinity。
        def clean(value):
            if isinstance(value, float) and not math.isfinite(value): return None
            if isinstance(value, dict): return {k: clean(v) for k, v in value.items()}
            if isinstance(value, list): return [clean(v) for v in value]
            return value
        save_json(name + ("_profile" if profile_run else "") + ".json", clean(result))


In [ ]:
RESULTS = []
for name, micro, ckpt in CASES:
    print("开始策略：", name)
    RESULTS.append(run_case(name, micro, ckpt))
print("所有策略已结束；下一单元才做真实不变量判定。")


In [ ]:
baseline = next(r for r in RESULTS if r["strategy"] == "baseline")
base_ok = baseline["status"] == "COMPLETE" and len(baseline["steps"]) == N_STEPS
DECISIONS = []
for r in RESULTS:
    complete = r["status"] == "COMPLETE"
    checks = dict(completed=complete,
                  full_update_count=len(r["steps"]) == N_STEPS,
                  no_skipped_updates=bool(r["steps"]) and all(not s["skipped_update"] for s in r["steps"]),
                  finite_train_losses=bool(r["steps"]) and all(math.isfinite(s["loss"]) for s in r["steps"]),
                  baseline_available=base_ok)
    if complete and base_ok:
        checks.update(same_initial_identity=r["identity"] == baseline["identity"],
                      same_current_inputs=r["identity"]["train_sha256"] == tensor_sha(train_cpu)
                                          and r["identity"]["validation_sha256"] == tensor_sha(eval_cpu),
                      initial_validation_close=abs(r["initial_validation_loss"]-baseline["initial_validation_loss"]) <= CFG["initial_loss_atol"],
                      final_validation_close=abs(r["final_validation_loss"]-baseline["final_validation_loss"]) <= CFG["final_loss_atol"],
                      throughput_ok=r["target_tokens_per_second"] >= baseline["target_tokens_per_second"]*CFG["min_throughput_ratio"],
                      memory_budget_ok=r["train_peak_reserved_gib"]+CFG["safety_margin_gib"] <= r["budget"]["budget_gib"])
    decision = dict(strategy=r["strategy"], status=r["status"], verdict="PASS" if all(checks.values()) else "REJECT", checks=checks)
    DECISIONS.append(decision)
    print(json.dumps(decision, ensure_ascii=False, indent=2))
save_json("decisions.json", DECISIONS)
rows = [dict(strategy=r["strategy"], **step) for r in RESULTS for step in r["steps"]]
if rows:
    with (RUN / "steps.csv").open("w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)
completed = [r for r in RESULTS if r["status"] == "COMPLETE"]
if completed:
    names = [r["strategy"] for r in completed]
    x = np.arange(len(names))
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
    axes[0].bar(x-0.18, [r["train_peak_allocated_gib"] for r in completed], 0.36, label="allocated")
    axes[0].bar(x+0.18, [r["train_peak_reserved_gib"] for r in completed], 0.36, label="reserved")
    axes[0].axhline(min(CFG["budget_gib"], ENV["total_gib"])-CFG["safety_margin_gib"], color="red", linestyle="--", label="budget - margin")
    axes[0].set_ylabel("GiB (training only)")
    axes[0].legend()
    axes[1].bar(x, [r["target_tokens_per_second"] for r in completed], color="teal")
    axes[1].set_ylabel("target tokens / second")
    axes[2].bar(x-0.18, [r["initial_validation_loss"] for r in completed], 0.36, label="initial")
    axes[2].bar(x+0.18, [r["final_validation_loss"] for r in completed], 0.36, label="final")
    axes[2].set_ylabel("held-out loss (short run)")
    axes[2].legend()
    for ax in axes:
        ax.set_xticks(x, names, rotation=15)
    fig.tight_layout()
    fig.savefig(RUN / "comparison.png", dpi=160)
    plt.show()
    fig, ax = plt.subplots(figsize=(9, 4))
    for i, r in enumerate(completed):
        ax.plot([s["step"] for s in r["steps"]], [s["loss"] for s in r["steps"]], marker="o", linestyle=["-","--",":"][i % 3], label=r["strategy"] + " (curves overlap = updates are numerically equivalent)")
    ax.axvline(CFG["warmup_steps"]-0.5, linestyle="--", color="gray", label="measurement boundary")
    ax.set(xlabel="optimizer update", ylabel="training loss")
    ax.legend()
    fig.tight_layout()
    fig.savefig(RUN / "training_loss.png", dpi=160)
    plt.show()
else:
    print("没有完成的策略，不绘制虚构结果。请查看各策略拒绝原因。")


## 4.2 增项实测：账本 / offload / 独立 workload / 量化 / 预算敏感性 / profiler 分类

以下单元格全部是 GPU 实测，不用任何估算公式：

- **显存账本（实测）**：参数、梯度、优化器状态按张量 `numel × element_size` 精确求和；激活与临时缓冲 = 「反向结束时的 allocated − 参数 − 梯度」实测；运行时缓存 = 「reserved − allocated」实测。
- **activation offload（实测）**：`save_on_cpu` 包住前向，跑完整 warmup + measured，同数据、同初始权重，记录峰值、吞吐与 held-out loss。
- **独立 workload（实测）**：有效 batch 4→1、序列 128→64。工作负载改变，只比较显存与单位 token 吞吐，**不与主表比 loss**。
- **量化（实测）**：bitsandbytes 8-bit 优化器完整训练；4-bit 权重只做静态加载与前向实测（权重冻结，不训练，因此不与训练策略比吞吐）。
- **预算敏感性**：峰值与吞吐全部来自实测，只扫描预算阈值。
- **profiler 分类（实测 trace）**：导出的 chrome trace 中 `kernel` / `gpu_memcpy` / `cuda_runtime` 事件按类型聚合，区分计算、访存候选、数据传输与同步/调度等待。

注意执行顺序：先跑完主表（task3-08/09），再依次跑本页单元格；本页单元格依赖前面的全局函数与数据。
**排障（重要）**：若先跑过主表才在量化格 `%pip install bitsandbytes`，transformers 的 bnb 集成模块
会在缺 `nn`/`bnb` 的状态下被缓存，之后 `from_pretrained(load_in_4bit=True)` 连续报 NameError。
正确做法：修改安装格后 **运行时 → 重启会话 → 全部运行**，让 transformers 在 bitsandbytes 装好之后才被 import。
量化格内的缓存刷新与命名空间注入只是兜底；若重启并全部运行后 4-bit 加载仍失败，
说明该运行时版本组合不兼容——跳过 4-bit 静态测量并在证据中如实标注，不要伪造数字。


In [ ]:
# 4.2(2) 实测显存账本：常驻状态按张量精确求和；激活/临时用实测差值；缓存用 reserved-allocated
cleanup()
seed_all()
ledger_model, ledger_ident, ledger_budget = make_model(False)
ledger_opt = torch.optim.AdamW(ledger_model.parameters(), lr=CFG["lr"],
                               weight_decay=CFG["weight_decay"], foreach=False)
ledger_scaler = torch.amp.GradScaler("cuda", enabled=AMP_DTYPE == torch.float16, init_scale=1024.0)
sync()
points = dict(after_load_allocated_gib=torch.cuda.memory_allocated(DEVICE) / GIB)
torch.cuda.reset_peak_memory_stats(DEVICE)
start = time.perf_counter()
ledger_model.train()
ledger_opt.zero_grad(set_to_none=True)
x = train_batches[0].to(DEVICE)
with torch.autocast("cuda", dtype=AMP_DTYPE):
    loss = ledger_model(input_ids=x, attention_mask=torch.ones_like(x), labels=x, use_cache=False).loss
    normalized = loss * (x.shape[0] / CFG["effective_batch"])
ledger_scaler.scale(normalized).backward()
del x, loss, normalized
sync()
points["after_backward_allocated_gib"] = torch.cuda.memory_allocated(DEVICE) / GIB
ledger_scaler.unscale_(ledger_opt)
norm = torch.nn.utils.clip_grad_norm_(ledger_model.parameters(), CFG["max_grad_norm"], foreach=False)
ledger_scaler.step(ledger_opt)
ledger_scaler.update()
sync()
points["after_optimizer_allocated_gib"] = torch.cuda.memory_allocated(DEVICE) / GIB
points["after_step_reserved_gib"] = torch.cuda.memory_reserved(DEVICE) / GIB
points["peak_allocated_gib"] = torch.cuda.max_memory_allocated(DEVICE) / GIB
points["seconds"] = time.perf_counter() - start

param_gib = sum(p.numel() * p.element_size() for p in ledger_model.parameters()) / GIB
grad_gib = sum(p.grad.numel() * p.grad.element_size()
               for p in ledger_model.parameters() if p.grad is not None) / GIB
state_gib = sum(v.numel() * v.element_size()
                for st in ledger_opt.state.values() for v in st.values()
                if torch.is_tensor(v)) / GIB
resident_gib = points["after_optimizer_allocated_gib"]
LEDGER = dict(
    parameters_gib=round(param_gib, 4),
    gradients_gib=round(grad_gib, 4),
    optimizer_state_gib=round(state_gib, 4),
    model_state_gib=round(param_gib + grad_gib + state_gib, 4),
    measured_resident_allocated_gib=round(resident_gib, 4),
    activations_and_temporary_gib=round(points["after_backward_allocated_gib"] - param_gib - grad_gib, 4),
    peak_transient_above_resident_gib=round(points["peak_allocated_gib"] - resident_gib, 4),
    allocator_cache_gib=round(points["after_step_reserved_gib"] - resident_gib, 4),
    measurement_points={k: round(v, 4) for k, v in points.items()})
# 用实测常驻状态解释各策略的实测峰值
LEDGER["per_strategy_peak_minus_state_gib"] = {
    r["strategy"]: round(r["train_peak_allocated_gib"] - (param_gib + grad_gib + state_gib), 4)
    for r in RESULTS if r["status"] == "COMPLETE"}
save_json("memory_ledger.json", LEDGER)
print(json.dumps(LEDGER, ensure_ascii=False, indent=2))
del ledger_model, ledger_opt, ledger_scaler
cleanup()

In [ ]:
# 4.2(3) activation offload 实测：save_on_cpu 包住前向，完整 warmup+measured，同数据同初始权重
EXTRA_RESULTS = []

def _clean(value):
    if isinstance(value, float) and not math.isfinite(value):
        return None
    if isinstance(value, dict):
        return {k: _clean(v) for k, v in value.items()}
    if isinstance(value, list):
        return [_clean(v) for v in value]
    return value

def train_step_offload(model, optimizer, scaler, batch, micro, effective_batch=None):
    """与 train_step 完全同口径，唯一差别：前向在 save_on_cpu 上下文里执行。"""
    eff = CFG["effective_batch"] if effective_batch is None else effective_batch
    model.train()
    optimizer.zero_grad(set_to_none=True)
    sync()
    torch.cuda.reset_peak_memory_stats(DEVICE)
    start = time.perf_counter()
    losses = []
    for offset in range(0, eff, micro):
        x = batch[offset:offset + micro].to(DEVICE)
        # save_on_cpu 会把 autograd 保存的张量卸载到 CPU（含激活与保存的参数引用）
        with torch.autograd.graph.save_on_cpu(pin_memory=True):
            with torch.autocast("cuda", dtype=AMP_DTYPE):
                loss = model(input_ids=x, attention_mask=torch.ones_like(x), labels=x, use_cache=False).loss
                normalized = loss * (x.shape[0] / eff)
        scaler.scale(normalized).backward()
        losses.append(loss.detach() * (x.shape[0] / eff))
        del x, loss, normalized
    scaler.unscale_(optimizer)
    norm = torch.nn.utils.clip_grad_norm_(model.parameters(), CFG["max_grad_norm"], foreach=False)
    finite = bool(torch.isfinite(norm).item())
    old_scale = scaler.get_scale()
    if finite:
        scaler.step(optimizer)
    scaler.update()
    skipped = (not finite) or scaler.get_scale() < old_scale
    mean_loss = float(torch.stack(losses).sum())
    del losses, norm
    sync()
    elapsed = time.perf_counter() - start
    return dict(loss=mean_loss, seconds=elapsed,
                input_tokens_per_second=eff * batch.shape[-1] / elapsed,
                target_tokens_per_second=eff * (batch.shape[-1] - 1) / elapsed,
                peak_allocated_gib=torch.cuda.max_memory_allocated(DEVICE) / GIB,
                peak_reserved_gib=torch.cuda.max_memory_reserved(DEVICE) / GIB,
                allocated_gib=torch.cuda.memory_allocated(DEVICE) / GIB,
                reserved_gib=torch.cuda.memory_reserved(DEVICE) / GIB,
                scale_before=old_scale, scale_after=scaler.get_scale(), skipped_update=skipped)

def run_offload_case(name="activation_offload"):
    cleanup()
    model, ident, budget = make_model(False)
    optimizer = torch.optim.AdamW(model.parameters(), lr=CFG["lr"],
                                  weight_decay=CFG["weight_decay"], foreach=False)
    scaler = torch.amp.GradScaler("cuda", enabled=AMP_DTYPE == torch.float16, init_scale=1024.0)
    result = dict(strategy=name, workload="main:seq%d/eff%d" % (CFG["seq_len"], CFG["effective_batch"]),
                  micro_batch=CFG["effective_batch"], accumulation=1, checkpoint=False, offload=True,
                  status="RUNNING", steps=[])
    try:
        result.update(identity=ident, budget=budget)
        result["initial_validation_loss"] = evaluate(model, optimizer)
        cleanup()
        for i, batch in enumerate(train_batches):
            if i == CFG["warmup_steps"]:
                optimizer.zero_grad(set_to_none=True)
                sync()
                torch.cuda.reset_peak_memory_stats(DEVICE)
            record = train_step_offload(model, optimizer, scaler, batch, CFG["effective_batch"])
            record.update(step=i, phase="warmup" if i < CFG["warmup_steps"] else "measured")
            result["steps"].append(record)
            print(name, record)
            if record["skipped_update"] or not math.isfinite(record["loss"]):
                result["status"] = "REJECT_SKIPPED_OR_NONFINITE"
                return result
        measured = [r for r in result["steps"] if r["phase"] == "measured"]
        result.update(train_peak_allocated_gib=max(r["peak_allocated_gib"] for r in measured),
                      train_peak_reserved_gib=max(r["peak_reserved_gib"] for r in measured),
                      target_tokens_per_second=len(measured) * INPUTS["causal_targets_per_update"] / sum(r["seconds"] for r in measured),
                      step_seconds_median=statistics.median(r["seconds"] for r in measured))
        optimizer.zero_grad(set_to_none=True)
        result["final_validation_loss"] = evaluate(model, optimizer)
        result["status"] = "COMPLETE" if math.isfinite(result["final_validation_loss"]) else "REJECT_NONFINITE"
        return result
    finally:
        if optimizer is not None:
            optimizer.zero_grad(set_to_none=True)
        model = optimizer = scaler = None
        cleanup()
        save_json(name + ".json", _clean(result))

offload_result = run_offload_case()
EXTRA_RESULTS.append(offload_result)
print("offload 状态：", offload_result["status"],
      "| 峰值 reserved GiB：", offload_result.get("train_peak_reserved_gib"),
      "| tokens/s：", offload_result.get("target_tokens_per_second"))

In [ ]:
# 4.2(3) 独立 workload 实测：缩小 batch 与缩短序列。工作负载改变，只比显存与吞吐，不与主表比 loss。
def repack(tokens_tensor, n_chunks, seq):
    flat = tokens_tensor.reshape(-1)
    need = n_chunks * seq
    assert flat.numel() >= need, "token 不足，不能复用数据伪造验证集"
    return flat[:need].reshape(n_chunks, seq)

def run_variant(name, batches, eval_tokens, effective_batch, micro, note):
    cleanup()
    model, ident, budget = make_model(False)
    optimizer = torch.optim.AdamW(model.parameters(), lr=CFG["lr"],
                                  weight_decay=CFG["weight_decay"], foreach=False)
    scaler = torch.amp.GradScaler("cuda", enabled=AMP_DTYPE == torch.float16, init_scale=1024.0)

    def eval_variant():
        model.eval()
        total, targets = 0.0, 0
        with torch.no_grad():
            for row in eval_tokens:
                xv = row.unsqueeze(0).to(DEVICE)
                with torch.autocast("cuda", dtype=AMP_DTYPE):
                    lv = model(input_ids=xv, attention_mask=torch.ones_like(xv), labels=xv, use_cache=False).loss
                n = xv.shape[0] * (xv.shape[1] - 1)
                total += float(lv) * n
                targets += n
                del xv, lv
        sync()
        model.train()
        return total / targets

    seq = batches.shape[-1]
    result = dict(strategy=name, workload="variant:seq%d/eff%d" % (seq, effective_batch),
                  micro_batch=micro, accumulation=effective_batch // micro, checkpoint=False, offload=False,
                  effective_batch=effective_batch, seq_len=seq, status="RUNNING", steps=[], note=note)
    try:
        initial = eval_variant()
        cleanup()
        for i, batch in enumerate(batches):
            if i == CFG["warmup_steps"]:
                optimizer.zero_grad(set_to_none=True)
                sync()
                torch.cuda.reset_peak_memory_stats(DEVICE)
            record = train_step(model, optimizer, scaler, batch, micro, effective_batch=effective_batch)
            record.update(step=i, phase="warmup" if i < CFG["warmup_steps"] else "measured")
            result["steps"].append(record)
            print(name, record)
            if record["skipped_update"] or not math.isfinite(record["loss"]):
                result["status"] = "REJECT_SKIPPED_OR_NONFINITE"
                return result
        measured = [r for r in result["steps"] if r["phase"] == "measured"]
        result.update(initial_validation_loss=initial,
                      train_peak_allocated_gib=max(r["peak_allocated_gib"] for r in measured),
                      train_peak_reserved_gib=max(r["peak_reserved_gib"] for r in measured),
                      target_tokens_per_second=len(measured) * effective_batch * (seq - 1) / sum(r["seconds"] for r in measured),
                      step_seconds_median=statistics.median(r["seconds"] for r in measured))
        optimizer.zero_grad(set_to_none=True)
        result["final_validation_loss"] = eval_variant()
        result["status"] = "COMPLETE" if math.isfinite(result["final_validation_loss"]) else "REJECT_NONFINITE"
        return result
    finally:
        if optimizer is not None:
            optimizer.zero_grad(set_to_none=True)
        model = optimizer = scaler = None
        cleanup()
        save_json(name + ".json", _clean(result))

# 同一 token 流重新切块：有效 batch=1 与 seq=64；数据来源与主表相同
small_batch_batches = repack(train_cpu, N_STEPS, CFG["seq_len"]).unsqueeze(1)
small_batch_eval = repack(eval_cpu, CFG["eval_chunks"], CFG["seq_len"])
short_seq_batches = repack(train_cpu, N_STEPS * CFG["effective_batch"], 64).reshape(N_STEPS, CFG["effective_batch"], 64)
short_seq_eval = repack(eval_cpu, CFG["eval_chunks"], 64)

EXTRA_RESULTS.append(run_variant("small_batch_eff1", small_batch_batches, small_batch_eval,
                                 effective_batch=1, micro=1,
                                 note="有效 batch 4→1：工作负载改变，只比显存与吞吐，不与主表比 loss"))
EXTRA_RESULTS.append(run_variant("short_seq64", short_seq_batches, short_seq_eval,
                                 effective_batch=CFG["effective_batch"], micro=CFG["effective_batch"],
                                 note="序列 128→64：工作负载改变，只比显存与吞吐，不与主表比 loss"))
for r in EXTRA_RESULTS:
    print(r["strategy"], r["status"], r.get("train_peak_reserved_gib"), r.get("target_tokens_per_second"))

In [ ]:
# 4.2(3) 量化实测：bitsandbytes 8-bit 优化器完整训练 + 4-bit 权重静态加载/前向实测%pip -q install "bitsandbytes>=0.43.0"import bitsandbytes as bnb# 本会话若在安装 bitsandbytes 之前已 import transformers，其可用性标志仍是 False，# from_pretrained(load_in_4bit=True) 会误报 requires the latest version of bitsandbytes；此处手动刷新。import transformers.utils.import_utils as _iu_iu._bitsandbytes_available = Trueimport transformers.quantizers.quantizer_bnb_4bit as _qbnbif hasattr(_qbnb, "is_bitsandbytes_available"):    _qbnb.is_bitsandbytes_available = lambda: Trueprint("bitsandbytes:", bnb.__version__, "| transformers availability cache refreshed")assert torch.cuda.is_available(), "量化实测需要 GPU"def train_step_bnb(model, optimizer, scaler, batch, micro, effective_batch=None):    """与 train_step 同口径；8-bit 优化器不依赖 GradScaler.unscale_，手动反缩放。"""    eff = CFG["effective_batch"] if effective_batch is None else effective_batch    model.train()    optimizer.zero_grad(set_to_none=True)    sync()    torch.cuda.reset_peak_memory_stats(DEVICE)    start = time.perf_counter()    losses = []    for offset in range(0, eff, micro):        x = batch[offset:offset + micro].to(DEVICE)        with torch.autocast("cuda", dtype=AMP_DTYPE):            loss = model(input_ids=x, attention_mask=torch.ones_like(x), labels=x, use_cache=False).loss            normalized = loss * (x.shape[0] / eff)        scaler.scale(normalized).backward()        losses.append(loss.detach() * (x.shape[0] / eff))        del x, loss, normalized    if scaler.is_enabled():        inv_scale = 1.0 / scaler.get_scale()        for p in model.parameters():            if p.grad is not None:                p.grad.mul_(inv_scale)    norm = torch.nn.utils.clip_grad_norm_(model.parameters(), CFG["max_grad_norm"], foreach=False)    finite = bool(torch.isfinite(norm).item())    old_scale = scaler.get_scale()    if finite:        optimizer.step()    scaler.update()    skipped = (not finite) or scaler.get_scale() < old_scale    mean_loss = float(torch.stack(losses).sum())    del losses, norm    sync()    elapsed = time.perf_counter() - start    return dict(loss=mean_loss, seconds=elapsed,                input_tokens_per_second=eff * batch.shape[-1] / elapsed,                target_tokens_per_second=eff * (batch.shape[-1] - 1) / elapsed,                peak_allocated_gib=torch.cuda.max_memory_allocated(DEVICE) / GIB,                peak_reserved_gib=torch.cuda.max_memory_reserved(DEVICE) / GIB,                allocated_gib=torch.cuda.memory_allocated(DEVICE) / GIB,                reserved_gib=torch.cuda.memory_reserved(DEVICE) / GIB,                scale_before=old_scale, scale_after=scaler.get_scale(), skipped_update=skipped)def run_quantized_optimizer_case(name="quant_adamw8bit"):    cleanup()    model, ident, budget = make_model(False)    # 8-bit AdamW：优化器状态从 fp32 两态压到 8-bit，这里实测压了多少    optimizer = bnb.optim.AdamW8bit(model.parameters(), lr=CFG["lr"],                                    weight_decay=CFG["weight_decay"], eps=1e-8)    scaler = torch.amp.GradScaler("cuda", enabled=AMP_DTYPE == torch.float16, init_scale=1024.0)    result = dict(strategy=name, workload="main:seq%d/eff%d" % (CFG["seq_len"], CFG["effective_batch"]),                  micro_batch=CFG["effective_batch"], accumulation=1, checkpoint=False, offload=False,                  quantized_optimizer="adamw8bit", status="RUNNING", steps=[])    try:        result.update(identity=ident, budget=budget)        result["initial_validation_loss"] = evaluate(model, optimizer)        cleanup()        for i, batch in enumerate(train_batches):            if i == CFG["warmup_steps"]:                optimizer.zero_grad(set_to_none=True)                sync()                torch.cuda.reset_peak_memory_stats(DEVICE)            record = train_step_bnb(model, optimizer, scaler, batch, CFG["effective_batch"])            record.update(step=i, phase="warmup" if i < CFG["warmup_steps"] else "measured")            result["steps"].append(record)            print(name, record)            if record["skipped_update"] or not math.isfinite(record["loss"]):                result["status"] = "REJECT_SKIPPED_OR_NONFINITE"                return result        measured = [r for r in result["steps"] if r["phase"] == "measured"]        state_gib = sum(v.numel() * v.element_size()                        for st in optimizer.state.values() for v in st.values()                        if torch.is_tensor(v)) / GIB        result.update(train_peak_allocated_gib=max(r["peak_allocated_gib"] for r in measured),                      train_peak_reserved_gib=max(r["peak_reserved_gib"] for r in measured),                      optimizer_state_gib=round(state_gib, 4),                      target_tokens_per_second=len(measured) * INPUTS["causal_targets_per_update"] / sum(r["seconds"] for r in measured),                      step_seconds_median=statistics.median(r["seconds"] for r in measured))        optimizer.zero_grad(set_to_none=True)        result["final_validation_loss"] = evaluate(model, optimizer)        result["status"] = "COMPLETE" if math.isfinite(result["final_validation_loss"]) else "REJECT_NONFINITE"        return result    finally:        if optimizer is not None:            optimizer.zero_grad(set_to_none=True)        model = optimizer = scaler = None        cleanup()        save_json(name + ".json", _clean(result))EXTRA_RESULTS.append(run_quantized_optimizer_case())# 4-bit 权重静态实测：只测加载显存与前向；权重冻结，不训练，因此不与训练策略比吞吐from transformers import BitsAndBytesConfigcleanup()seed_all()# transformers 的 bnb 集成在本会话启动时就已 import，那时 bitsandbytes 尚未安装，# 导入块整段跳过，之后无论怎么装包、怎么重载，旧的类/函数对象仍持有没有 nn/bnb 的旧字典。# 做法：先刷新可用性标志，再用 gc 扫描所有函数，凡 __globals__ 属于 bnb 相关旧字典的一律补齐。import gc as _gc, sys as _sys, types as _typesimport torch as _torch_p, torch.nn as _nn_p, torch.nn.functional as _F_pimport bitsandbytes as _bnb_pdef _ensure_dict(g):    for _k, _v in [("torch", _torch_p), ("nn", _nn_p), ("F", _F_p), ("bnb", _bnb_p)]:        if _k not in g:            g[_k] = _v    if "Conv1D" not in g:        try:            from transformers.pytorch_utils import Conv1D as _C1D            g["Conv1D"] = _C1D        except Exception:            g["Conv1D"] = _nn_p.Linear# 1) 可用性标志import transformers.utils.import_utils as _iu_p_iu_p._bitsandbytes_available = True# 2) 删掉相关模块重新 import，让新模块的导入块完整执行import importlib as _importlibfor _n in [k for k in list(_sys.modules) if k.startswith("transformers") and "bitsandbytes" in k]:    del _sys.modules[_n]_b = _importlib.import_module("transformers.integrations.bitsandbytes")_q = _importlib.import_module("transformers.quantizers.quantizer_bnb_4bit")_ensure_dict(_b.__dict__)_ensure_dict(_q.__dict__)# 3) gc 扫描：补齐所有仍指向旧字典的函数（lazy 缓存、已 import 的 quantizer.py 等都覆盖）_TARGETS = {"transformers.integrations.bitsandbytes", "transformers.quantizers.quantizer_bnb_4bit"}_fixed = 0for _obj in _gc.get_objects():    if isinstance(_obj, _types.FunctionType):        _g = _obj.__globals__        if _g.get("__name__") in _TARGETS:            _ensure_dict(_g)            _fixed += 1# 4) 已 import 模块的命名空间（普通模块，不在 lazy 缓存里）for _mod in list(_sys.modules.values()):    if getattr(_mod, "__name__", "") in _TARGETS:        _ensure_dict(_mod.__dict__)print("bnb integration ready:", {k: hasattr(_b, k) for k in ["torch", "nn", "F", "bnb", "Conv1D"]},      "| stale function dicts fixed:", _fixed)bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",                                bnb_4bit_compute_dtype=AMP_DTYPE, bnb_4bit_use_double_quant=True)q4_model = AutoModelForCausalLM.from_pretrained(MODEL_PATH, local_files_only=True,                                                quantization_config=bnb_config,                                                attn_implementation="sdpa", low_cpu_mem_usage=True)q4_model.config.use_cache = Falseq4_model.eval()sync()q4_static_gib = torch.cuda.memory_allocated(DEVICE) / GIBtorch.cuda.reset_peak_memory_stats(DEVICE)q4_loss = evaluate(q4_model)q4_peak_gib = torch.cuda.max_memory_allocated(DEVICE) / GIBstored_weight_gib = sum(p.numel() * p.element_size() for p in q4_model.parameters()) / GIBQ4 = dict(strategy="weight_nf4_static", workload="inference-only",          static_allocated_gib=round(q4_static_gib, 4),          forward_peak_allocated_gib=round(q4_peak_gib, 4),          stored_weight_bytes_gib=round(stored_weight_gib, 4),          heldout_loss=round(q4_loss, 6),          note="4-bit 权重冻结，只测加载与前向显存；不训练，因此不与训练策略比吞吐")save_json("weight_nf4_static.json", Q4)print(json.dumps(Q4, ensure_ascii=False, indent=2))del q4_modelcleanup()

In [ ]:
# 4.2(1)(3) 汇总：实测峰值/吞吐/loss 权衡表 + 预算阈值敏感性（峰值实测，只扫阈值）
import pandas as pd

try:
    EXTRA_RESULTS
except NameError:
    EXTRA_RESULTS = []

ALL_RESULTS = list(RESULTS) + list(EXTRA_RESULTS)
rows = []
for r in ALL_RESULTS:
    if r["status"] != "COMPLETE":
        rows.append(dict(strategy=r["strategy"], workload=r.get("workload"), status=r["status"]))
        continue
    rows.append(dict(strategy=r["strategy"], workload=r.get("workload"), status=r["status"],
                     peak_reserved_gib=round(r["train_peak_reserved_gib"], 3),
                     peak_allocated_gib=round(r["train_peak_allocated_gib"], 3),
                     tokens_per_s=round(r["target_tokens_per_second"], 1),
                     initial_loss=round(r["initial_validation_loss"], 6),
                     final_loss=round(r["final_validation_loss"], 6)))
summary = pd.DataFrame(rows)
main_baseline = next(r for r in ALL_RESULTS if r["strategy"] == "baseline" and r["status"] == "COMPLETE")
summary["memory_saving_vs_baseline_pct"] = [
    round((1 - p / main_baseline["train_peak_reserved_gib"]) * 100, 2) if pd.notna(p) else None
    for p in summary["peak_reserved_gib"]]
summary["throughput_ratio_vs_baseline"] = [
    round(t / main_baseline["target_tokens_per_second"], 3) if pd.notna(t) else None
    for t in summary["tokens_per_s"]]
summary["final_loss_delta_vs_baseline"] = [
    round(f - main_baseline["final_validation_loss"], 6) if pd.notna(f) else None
    for f in summary["final_loss"]]
summary["same_workload_as_baseline"] = summary["workload"].fillna("").str.startswith("main")
display(summary)

# 预算敏感性：峰值与吞吐全部实测，只扫描预算阈值
MAIN = [r for r in ALL_RESULTS if str(r.get("workload", "")).startswith("main") and r["status"] == "COMPLETE"]
min_tps = main_baseline["target_tokens_per_second"] * CFG["min_throughput_ratio"]
budget_rows = []
for budget_gib in [6, 8, 10, 12, 14, 16, 20, 24]:
    eligible = [r for r in MAIN
                if r["train_peak_reserved_gib"] + CFG["safety_margin_gib"] <= budget_gib
                and r["target_tokens_per_second"] >= min_tps
                and abs(r["final_validation_loss"] - main_baseline["final_validation_loss"]) <= CFG["final_loss_atol"]]
    best = max(eligible, key=lambda r: r["target_tokens_per_second"]) if eligible else None
    budget_rows.append(dict(budget_gib=budget_gib, eligible=[r["strategy"] for r in eligible],
                            recommendation=best["strategy"] if best else "infeasible"))
BUDGET_SENSITIVITY = budget_rows
save_json("budget_sensitivity.json", BUDGET_SENSITIVITY)
display(pd.DataFrame(BUDGET_SENSITIVITY))

names = [r["strategy"] for r in MAIN]
x = np.arange(len(names))
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
axes[0].bar(x - 0.18, [r["train_peak_allocated_gib"] for r in MAIN], 0.36, label="allocated")
axes[0].bar(x + 0.18, [r["train_peak_reserved_gib"] for r in MAIN], 0.36, label="reserved")
axes[0].axhline(min(CFG["budget_gib"], ENV["total_gib"]) - CFG["safety_margin_gib"], color="red", linestyle="--", label="budget - margin")
axes[0].set_ylabel("GiB (training only)")
axes[0].legend()
axes[1].bar(x, [r["target_tokens_per_second"] for r in MAIN], color="teal")
axes[1].set_ylabel("target tokens / second")
axes[2].bar(x - 0.18, [r["initial_validation_loss"] for r in MAIN], 0.36, label="initial")
axes[2].bar(x + 0.18, [r["final_validation_loss"] for r in MAIN], 0.36, label="final")
axes[2].set_ylabel("held-out loss (short run)")
axes[2].legend()
for ax in axes:
    ax.set_xticks(x, names, rotation=15)
fig.tight_layout()
fig.savefig(RUN / "comparison_extended.png", dpi=160)
plt.show()

styles = ["-", "--", ":", "-.", (0, (3, 1, 1, 1))]
fig, ax = plt.subplots(figsize=(9, 4))
for i, r in enumerate(MAIN):
    ax.plot([s["step"] for s in r["steps"]], [s["loss"] for s in r["steps"]],
            marker="o", linestyle=styles[i % len(styles)], label=r["strategy"])
ax.axvline(CFG["warmup_steps"] - 0.5, linestyle="--", color="gray", label="measurement boundary")
ax.set(xlabel="optimizer update", ylabel="training loss",
       title="curves overlap = updates are numerically equivalent")
ax.legend()
fig.tight_layout()
fig.savefig(RUN / "training_loss_extended.png", dpi=160)
plt.show()

try:
    ledger_for_summary = LEDGER
except NameError:
    ledger_for_summary = None
save_json("summary.json", dict(
    note="所有峰值/吞吐/loss 均为实测；variants 工作负载不同，只比显存与吞吐；weight_nf4_static 只做静态/前向。",
    ledger=ledger_for_summary,
    summary=summary.where(pd.notna(summary), None).to_dict(orient="records"),
    budget_sensitivity=BUDGET_SENSITIVITY))
print("已保存 summary.json / budget_sensitivity.json / comparison_extended.png / training_loss_extended.png")

In [ ]:
# 4.2(4) profiler trace 实测分类：kernel / gpu_memcpy / cuda_runtime 聚合，区分计算、访存、传输、同步等待
from collections import defaultdict

def profile_one_step(step_fn, name):
    cleanup()
    model, ident, budget = make_model(False)
    optimizer = torch.optim.AdamW(model.parameters(), lr=CFG["lr"],
                                  weight_decay=CFG["weight_decay"], foreach=False)
    scaler = torch.amp.GradScaler("cuda", enabled=AMP_DTYPE == torch.float16, init_scale=1024.0)
    model.train()
    optimizer.zero_grad(set_to_none=True)
    with torch.profiler.profile(activities=[torch.profiler.ProfilerActivity.CPU,
                                            torch.profiler.ProfilerActivity.CUDA],
                                record_shapes=True, profile_memory=True) as prof:
        record = step_fn(model, optimizer, scaler, train_batches[0], CFG["effective_batch"])
    path = RUN / (name + "_trace.json")
    prof.export_chrome_trace(str(path))
    model = optimizer = scaler = None
    cleanup()
    return path, record

baseline_trace, baseline_profile_record = profile_one_step(train_step, "profiled_baseline")
offload_trace, offload_profile_record = profile_one_step(train_step_offload, "profiled_offload")

def classify_event(event):
    cat = event.get("cat", "")
    name = event.get("name", "")
    low = name.lower()
    if cat == "gpu_memcpy":
        return "数据传输(memcpy)"
    if cat == "gpu_memset":
        return "数据传输(memset)"
    if cat == "cuda_runtime":
        if "synchronize" in low or "streamwait" in low:
            return "同步等待(runtime)"
        return "调度/启动(runtime)"
    if cat == "kernel":
        if any(k in low for k in ["gemm", "splitk", "cutlass", "sgemm", "gemv"]):
            return "计算候选(gemm kernel)"
        if any(k in low for k in ["elementwise", "vectorized", "reduce", "copy", "transpose", "softmax", "layernorm", "norm"]):
            return "访存候选(elementwise/reduce kernel)"
        return "计算候选(其他 kernel)"
    if cat == "cpu_op":
        return "CPU 算子"
    if cat == "cpu_instant":
        return "CPU 瞬时事件"
    return cat or "其他"

def summarize_trace(path):
    events = json.loads(path.read_text(encoding="utf-8"))["traceEvents"]
    agg = defaultdict(lambda: dict(count=0, total_us=0.0))
    top = defaultdict(lambda: defaultdict(lambda: [0, 0.0]))
    for e in events:
        if e.get("ph") != "X" or "dur" not in e:
            continue
        kind = classify_event(e)
        agg[kind]["count"] += 1
        agg[kind]["total_us"] += e.get("dur", 0.0)
        entry = top[kind][e.get("name", "unknown")]
        entry[0] += 1
        entry[1] += e.get("dur", 0.0)
    return dict(path=str(path), total_events=len(events),
                by_kind={k: dict(count=v["count"], total_us=round(v["total_us"], 1))
                         for k, v in sorted(agg.items(), key=lambda kv: -kv[1]["total_us"])},
                top_names={k: sorted(((nm, cnt, round(us, 1)) for nm, (cnt, us) in v.items()),
                                     key=lambda t: -t[2])[:6]
                           for k, v in top.items()})

TRACE_SUMMARY = dict(baseline=summarize_trace(baseline_trace),
                     offload=summarize_trace(offload_trace))
save_json("trace_summary.json", TRACE_SUMMARY)
print(json.dumps(TRACE_SUMMARY, ensure_ascii=False, indent=2))
print("baseline profile step:", baseline_profile_record)
print("offload profile step:", offload_profile_record)
print("offload 相比 baseline 的 memcpy 变化见 by_kind；trace 文件可直接在 Chrome://tracing 或 Perfetto 打开")

图轴使用英文以避免 Colab 默认字体缺失中文字形；解释与结论以本页中文及 decisions.json 为准。图只展示完成的策略，即使它被预算或质量规则拒绝；柱子的出现不表示 PASS。预热之后的缓存仍保留，因此 reserved 峰值包含正常缓存占用。

## 可选 profiler（与主实验隔离）

在配置中将 run_profiler 改为 True 后，执行下面单元。它重新加载同一初态，显式启用 checkpoint，先做同样的预热，再记录一个训练更新。profiler 的额外内存和时间不进入主表。trace 可在 Perfetto 或 Chrome tracing 中打开。此步骤也可能 OOM，不影响已保存的主实验结果。


In [ ]:
if CFG["run_profiler"]:
    profile_result = run_case("checkpoint", 2, True, profile_run=True)
    print("独立 profiler 状态：", profile_result["status"])
    trace = RUN / "checkpoint_trace.json"
    if trace.exists():
        from google.colab import files
        files.download(str(trace))
else:
    print("可选 profiler 未启用。")


## 导出与提交

下面下载 zip，包括版本、配置、模型/数据提交 SHA、固定输入、每种策略原始逐步记录、判定、CSV 和实测图（若有）。不会打包庞大的模型缓存。请另外选择 **文件 → 下载 → 下载 .ipynb** 保存含实际输出的已执行 notebook；运行时无法可靠获取浏览器中当前 notebook 的完整输出，所以 zip 不伪造已执行 notebook。可另存一份到 Google Drive。临时运行时断开后 /content 会丢失。

提交前检查：环境 GPU 和精度；训练/验证提交；所有策略相同初态与输入；真实 warmup/measured 记录；显存单位 GiB；拒绝原因；短跑质量代理的局限。出现普通 Python/网络/依赖错误时修复后重新运行，不能把它们当作 OOM 跳过。模型下载、真实 GPU 执行和数值稳定性必须由实际 Colab 运行确认。


In [ ]:
save_json("export_manifest.json", dict(files=sorted(p.name for p in RUN.iterdir()),
                                      executed_notebook="请在 Colab 文件菜单单独下载含输出的 .ipynb",
                                      scope="真实模型短跑，非收敛结论；仅本次硬件与配置有效"))
archive = shutil.make_archive(str(RUN), "zip", root_dir=RUN)
print("证据压缩包：", archive)
from google.colab import files
files.download(archive)
